In [1]:
import os
import sys
import importlib
import subprocess

print('Notebook kernel executable:', sys.executable)
print('Python version:', sys.version)

# Import core TensorFlow components
import tensorflow as tf
import keras

print('Imported TensorFlow version:', tf.__version__)
print('Imported Keras version:', keras.__version__)

# Ensure tf.keras is available
if not hasattr(tf, 'keras'):
    tf.keras = keras

from packaging import version
print('Setup complete: TensorFlow', tf.__version__, 'and Keras', keras.__version__, 'ready')


Notebook kernel executable: c:\Users\Mohith Kumar R\AppData\Local\Programs\Python\Python310\python.exe
Python version: 3.10.0 (tags/v3.10.0:b494f59, Oct  4 2021, 19:00:18) [MSC v.1929 64 bit (AMD64)]

Imported TensorFlow version: 2.15.1
Imported Keras version: 2.15.0
Setup complete: TensorFlow 2.15.1 and Keras 2.15.0 ready


In [2]:
try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except Exception:
    print("Not running in Colab; skipping drive.mount().")


Not running in Colab; skipping drive.mount().


In [3]:
import sys
%pip install -q Flask

Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install -q pyngrok nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install -q pyngrok nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install -q pyngrok nest_asyncio

Note: you may need to restart the kernel to use updated packages.


In [7]:
# 1. Install required packages
import sys
%pip install -q scikit-learn
%pip install -q pandas

import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

# 2. Load the IMDB dataset using Keras built-in loader (avoids tensorflow_datasets protobuf issues)
print("Loading IMDB dataset using Keras built-in loader...")
(X_train_raw, y_train), (X_test_raw, y_test) = tf.keras.datasets.imdb.load_data()

# Get word index to convert integers back to words
word_index = tf.keras.datasets.imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

def decode_review(encoded_review):
    """Convert integer-encoded review back to text"""
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Convert to dataframe with reviews and sentiments
reviews = [decode_review(review) for review in X_train_raw]
sentiments = ['positive' if label == 1 else 'negative' for label in y_train]
df_train_from_keras = pd.DataFrame({'review': reviews, 'sentiment': sentiments})

reviews_test = [decode_review(review) for review in X_test_raw]
sentiments_test = ['positive' if label == 1 else 'negative' for label in y_test]
df_test_from_keras = pd.DataFrame({'review': reviews_test, 'sentiment': sentiments_test})

print(f"Training set: {len(df_train_from_keras)} reviews")
print(f"Test set: {len(df_test_from_keras)} reviews")

# 3. Split into train/valid/test
train_set, valid_set = train_test_split(
    df_train_from_keras,
    test_size=0.1,
    stratify=df_train_from_keras['sentiment'],
    random_state=42
)

test_set = df_test_from_keras

print("Dataset split complete!")
print(f"Train: {len(train_set)}, Valid: {len(valid_set)}, Test: {len(test_set)}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Loading IMDB dataset using Keras built-in loader...
Training set: 25000 reviews
Test set: 25000 reviews
Dataset split complete!
Train: 22500, Valid: 2500, Test: 25000


In [1]:
def df_to_tf_dataset(df, batch_size=32, training=False):
    reviews = tf.constant(df['review'].values, dtype=tf.string)
    labels = tf.constant((df['sentiment'].values == 'positive').astype(int), dtype=tf.int64)

    ds = tf.data.Dataset.from_tensor_slices((reviews, labels))

    if training:
        ds = ds.shuffle(5000)

    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)
    return ds


In [9]:
if 'train_set' not in globals() or 'valid_set' not in globals() or 'test_set' not in globals():
    raise RuntimeError('Dataset variables are not defined. Run the dataset loading/preparation cell above before this one.')

train_set = df_to_tf_dataset(train_set, batch_size=32, training=True)
valid_set = df_to_tf_dataset(valid_set, batch_size=32)
test_set  = df_to_tf_dataset(test_set, batch_size=32)

In [10]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))


In [11]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)


Epoch 1/2

704/704 [==============================] - 221s 311ms/step - loss: 0.6934 - accuracy: 0.5038 - val_loss: 0.6932 - val_accuracy: 0.4956
Epoch 2/2
704/704 [==============================] - 209s 297ms/step - loss: 0.6924 - accuracy: 0.5067 - val_loss: 0.6932 - val_accuracy: 0.5032


In [12]:
embed_size = 128
tf.random.set_seed(42)
model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size,mask_zero = True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])
model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=5)

Epoch 1/5
704/704 [==============================] - 253s 355ms/step - loss: 0.5157 - accuracy: 0.7390 - val_loss: 0.4260 - val_accuracy: 0.8184
Epoch 2/5
704/704 [==============================] - 253s 360ms/step - loss: 0.3822 - accuracy: 0.8322 - val_loss: 0.3598 - val_accuracy: 0.8512
Epoch 3/5
704/704 [==============================] - 251s 357ms/step - loss: 0.2922 - accuracy: 0.8788 - val_loss: 0.2999 - val_accuracy: 0.8756
Epoch 4/5
704/704 [==============================] - 250s 355ms/step - loss: 0.2580 - accuracy: 0.8970 - val_loss: 0.3058 - val_accuracy: 0.8688
Epoch 5/5
704/704 [==============================] - 253s 359ms/step - loss: 0.2373 - accuracy: 0.9050 - val_loss: 0.3282 - val_accuracy: 0.8620


In [13]:
text_vec_layer_ragged = tf.keras.layers.TextVectorization(
    max_tokens=vocab_size, ragged=True)
text_vec_layer_ragged.adapt(train_set.map(lambda reviews, labels: reviews))

In [14]:
embed_size = 128
tf.random.set_seed(42)

model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size, mask_zero=True),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="nadam",
              metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=5)

Epoch 1/5
704/704 [==============================] - 260s 366ms/step - loss: 0.5663 - accuracy: 0.6958 - val_loss: 0.5585 - val_accuracy: 0.7428
Epoch 2/5
704/704 [==============================] - 272s 386ms/step - loss: 0.3729 - accuracy: 0.8407 - val_loss: 0.3525 - val_accuracy: 0.8544
Epoch 3/5
704/704 [==============================] - 274s 388ms/step - loss: 0.3037 - accuracy: 0.8742 - val_loss: 0.3657 - val_accuracy: 0.8428
Epoch 4/5
704/704 [==============================] - 270s 383ms/step - loss: 0.2965 - accuracy: 0.8791 - val_loss: 0.3107 - val_accuracy: 0.8712
Epoch 5/5
704/704 [==============================] - 267s 379ms/step - loss: 0.2488 - accuracy: 0.9009 - val_loss: 0.3053 - val_accuracy: 0.8732


In [15]:
import os
import tensorflow_hub as hub
import tensorflow as tf

os.environ["TFHUB_CACHE_DIR"] = "my_tfhub_cache"

class USELayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )

    def call(self, inputs):
        return self.use_layer(inputs)

use_layer = USELayer()

text_input = tf.keras.layers.Input(shape=[], dtype="string", name="text")
embedding = use_layer(text_input)
dense = tf.keras.layers.Dense(128, activation="relu")(embedding)
output = tf.keras.layers.Dense(1, activation="sigmoid")(dense)
model = tf.keras.Model(inputs=text_input, outputs=output)

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])  
model.fit(train_set, validation_data=valid_set, epochs=10)

C:\Users\Mohith Kumar R\AppData\Roaming\Python\Python310\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Epoch 1/10
704/704 [==============================] - 2222s 3s/step - loss: 0.2888 - accuracy: 0.8793 - val_loss: 0.2757 - val_accuracy: 0.8868
Epoch 2/10
704/704 [==============================] - 2199s 3s/step - loss: 0.0263 - accuracy: 0.9930 - val_loss: 0.3109 - val_accuracy: 0.9016
Epoch 3/10
704/704 [==============================] - 2200s 3s/step - loss: 0.0022 - accuracy: 0.9997 - val_loss: 0.4055 - val_accuracy: 0.8964
Epoch 4/10
704/704 [==============================] - 2200s 3s/step - loss: 5.2896e-04 - accuracy: 1.0000 - val_loss: 0.4426 - val_accuracy: 0.8964
Epoch 5/10
704/704 [==============================] - 2204s 3s/step - loss: 3.4611e-04 - accuracy: 1.0000 - val_loss: 0.4725 - val_accuracy: 0.8968
Epoch 6/10
704/704 [==============================] - 2210s 3s/step - loss: 3.2141e-05 - accuracy: 1.0000 - val_loss: 0.5265 - val_accuracy: 0.8968
Epoch 7/10
704/704 [==============================] - 2211s 3s/step - loss: 1.4403e-05 - accuracy: 1.0000 - val_loss: 0.5839

In [16]:
import os
import tensorflow as tf
import tensorflow_hub as hub

# 1. Define the architecture



class USELayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )

    def call(self, inputs):
        return self.use_layer(inputs)

# Re-initialize the model since it might have been cleared from memory
use_layer = USELayer()
text_input = tf.keras.layers.Input(shape=[], dtype="string", name="text")
embedding = use_layer(text_input)
dense = tf.keras.layers.Dense(128, activation="relu")(embedding)
output = tf.keras.layers.Dense(1, activation="sigmoid")(dense)
model = tf.keras.Model(inputs=text_input, outputs=output)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# 2. Save the model to a confirmed path
model_path = os.path.join(os.getcwd(), "movie_sentiment_model.keras")
print(f"Saving model to {model_path}...")
model.save(model_path)

if os.path.exists(model_path):
    print(f"✅ SUCCESS: Model saved at {model_path}")
else:
    print("❌ ERROR: Model file creation failed.")

Saving model to c:\Users\Mohith Kumar R\Downloads\archive\movie_sentiment_model.keras...
✅ SUCCESS: Model saved at c:\Users\Mohith Kumar R\Downloads\archive\movie_sentiment_model.keras


Now, let's create a Flask application to serve a simple web page for movie review sentiment prediction. This app will load the saved model, provide an input form, and display the prediction result.

In [17]:
import sys
import os

# 1. Force install dependencies to ensure they are available in this session
%pip install -q Flask pyngrok nest_asyncio

import tensorflow as tf
import tensorflow_hub as hub
from flask import Flask, request, render_template_string
from threading import Thread
import nest_asyncio

# Now import pyngrok after installation
try:
    from pyngrok import ngrok, conf
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyngrok'])
    from pyngrok import ngrok, conf

# 2. Define Custom Layer for Model Loading
class USELayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.use_layer = hub.KerasLayer(
            "https://tfhub.dev/google/universal-sentence-encoder/4",
            trainable=True,
            name="USE_embedding"
        )

    def call(self, inputs):
        return self.use_layer(inputs)

app = Flask(__name__)
app.config['ENV'] = 'development'
app.config['DEBUG'] = False

# 3. Load the saved model globally using the verified absolute path
model_path = os.path.join(os.getcwd(), "movie_sentiment_model.keras")
model = None

if not os.path.exists(model_path):
    print(f"❌ Error: {model_path} not found.")
else:
    try:
        model = tf.keras.models.load_model(
            model_path,
            custom_objects={'USELayer': USELayer}
        )
        print("✅ Model loaded successfully from absolute path!")
    except Exception as e:
        print(f"❌ Error loading model: {e}")

Note: you may need to restart the kernel to use updated packages.


✅ Model loaded successfully from absolute path!


In [2]:
import os
import pandas as pd

# Load the dataset from the local CSV file
csv_path = os.path.join(os.getcwd(), 'IMDB Dataset.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"CSV file not found at {csv_path}")

df_preview = pd.read_csv(csv_path)

# Display the first few rows
display(df_preview.head())

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
